# Dataset Exploration


In [ ]:
!pip install datasets -q

### Explorer Helper


In [ ]:
import pandas as pd
import random
import hashlib
from datasets import load_dataset, get_dataset_split_names

def explore(path, name=None, split="train", n=3, sample=500, seed=42):
    print(f"{'='*60}")
    print(f"  {path}  |  config={name}")
    print(f"{'='*60}")

    try:
        splits = get_dataset_split_names(path, name) if name else get_dataset_split_names(path)
        print(f"  Available splits : {splits}")
    except Exception as e:
        print(f"  Could not fetch split names: {e}")

    print(f"  Loading split    : '{split}' ...")
    try:
        ds = load_dataset(path, name, split=split) if name else load_dataset(path, split=split)
    except Exception as e:
        print(f"  ERROR loading: {e}")
        return None

    df_full = ds.to_pandas()
    print(f"\n  ── Full dataset stats {'─'*38}")
    print(f"  Total rows  : {len(df_full)}")
    print(f"  Columns     : {df_full.columns.tolist()}")
    print()

    for col in df_full.columns:
      print(f"  [{col}]")
      print(f"    dtype      : {df_full[col].dtype}")
      print(f"    sample val : {repr(df_full[col].iloc[0])[:120]}")
      print(f"    nulls      : {df_full[col].isna().sum()}")
      try:
          n_unique = df_full[col].nunique()
          print(f"    n_unique   : {n_unique}")
          if n_unique <= 20:
              print(f"    counts     : {df_full[col].value_counts().to_dict()}")
      except TypeError:
          print(f"    n_unique   : unhashable (list/array column)")
      try:
          lengths = df_full[col].apply(len)
          print(f"    len dist   : {lengths.value_counts().to_dict()}")
      except (TypeError, AttributeError):
          pass
      print()

    print(f"  ── First {n} examples {'─'*38}")
    for i in range(min(n, len(df_full))):
        print(f"  ── Example {i} {'─'*40}")
        for k, v in ds[i].items():
            print(f"  {k:20s}: {repr(v)[:500]}")
        print()

    random.seed(seed)
    if sample is None:
        df_sampled = df_full.reset_index(drop=True)
    else:
        valid_indices = []
        for idx in range(len(df_full)):
            q = str(df_full.iloc[idx].get("question", ""))[:200].strip().lower()
            fp = hashlib.md5(q.encode()).hexdigest()
            if fp not in existing_fps:
                valid_indices.append(idx)
        print(f"  After dedup: {len(valid_indices)} available (excluded {len(df_full) - len(valid_indices)})")
        indices = random.sample(valid_indices, min(sample, len(valid_indices)))
        df_sampled = df_full.iloc[indices].reset_index(drop=True)
    print(f"  Returning sampled DataFrame — {len(df_sampled)} rows (seed={seed})")
    return df_sampled

In [ ]:
import hashlib

old_df = pd.read_csv("RouterReflector Combined Dataset.csv")  # upload this first
existing_fps = set()
for _, row in old_df.iterrows():
    fp = hashlib.md5(str(row["question"])[:200].strip().lower().encode()).hexdigest()
    existing_fps.add(fp)
print(f"Loaded {len(existing_fps)} existing question fingerprints for dedup")

Loaded 1480 existing question fingerprints for dedup


## MMLU
Link: https://huggingface.co/datasets/cais/mmlu



In [ ]:
df_mmlu = explore("cais/mmlu", name="all", split="auxiliary_train", n=3, sample=500)

  cais/mmlu  |  config=all


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

  Available splits : ['test', 'validation', 'dev', 'auxiliary_train']
  Loading split    : 'auxiliary_train' ...


...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]


  ── Full dataset stats ──────────────────────────────────────
  Total rows  : 99842
  Columns     : ['question', 'subject', 'choices', 'answer']

  [question]
    dtype      : object
    sample val : "Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thin
    nulls      : 0
    n_unique   : 98487
    len dist   : {54: 147, 38: 135, 47: 135, 46: 130, 55: 127, 59: 127, 52: 126, 57: 125, 41: 125, 56: 123, 58: 121, 51: 118, 37: 118, 63: 118, 40: 116, 49: 115, 48: 115, 44: 114, 60: 112, 45: 110, 39: 107, 42: 107, 53: 106, 43: 105, 50: 105, 36: 104, 61: 103, 1749: 103, 67: 102, 1686: 101, 1812: 100, 69: 100, 35: 99, 65: 98, 1648: 98, 1724: 97, 1773: 97, 62: 97, 1882: 96, 1665: 96, 1874: 95, 1760: 95, 1782: 94, 34: 94, 64: 94, 1653: 94, 1887: 94, 66: 94, 1897: 93, 1736: 93, 1728: 93, 1697: 92, 1828: 92, 1842: 92, 30: 92, 33: 91, 74: 91, 1649: 90, 1877: 90, 32: 90, 1591: 90, 1607: 89, 1715: 89, 1774: 88, 1768: 88, 1771: 88, 173

In [ ]:
# Number of questions per option count
df_mmlu["choices"].apply(len).value_counts()

,count
choices,
4,500


## AQuA-RAT
Math dataset, substitute for MathQA

In [ ]:
# AQuA-RAT — algebra word problems, MCQ with 5 options, clean Parquet format
df_aqua = explore("aqua_rat", name="raw", split="train", n=3, sample=500)

  aqua_rat  |  config=raw


README.md: 0.00B [00:00, ?B/s]

  Available splits : ['train', 'test', 'validation']
  Loading split    : 'train' ...


raw/train-00000-of-00001.parquet:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

raw/test-00000-of-00001.parquet:   0%|          | 0.00/74.0k [00:00<?, ?B/s]

raw/validation-00000-of-00001.parquet:   0%|          | 0.00/76.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/97467 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/254 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/254 [00:00<?, ? examples/s]


  ── Full dataset stats ──────────────────────────────────────
  Total rows  : 97467
  Columns     : ['question', 'options', 'rationale', 'correct']

  [question]
    dtype      : object
    sample val : "Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's ra
    nulls      : 0
    n_unique   : 80924
    len dist   : {122: 722, 172: 716, 120: 692, 102: 673, 111: 662, 104: 657, 100: 653, 124: 621, 134: 619, 113: 614, 103: 603, 123: 597, 153: 585, 154: 585, 114: 583, 126: 579, 164: 578, 87: 575, 162: 572, 112: 567, 144: 566, 107: 563, 125: 559, 127: 558, 141: 549, 101: 548, 118: 548, 158: 543, 121: 542, 129: 538, 96: 538, 163: 532, 117: 531, 147: 531, 145: 529, 165: 526, 99: 526, 178: 525, 131: 523, 157: 516, 146: 516, 88: 511, 173: 511, 139: 508, 108: 505, 128: 496, 135: 489, 168: 488, 137: 484, 167: 481, 174: 478, 93: 478, 156: 475, 138: 469, 115: 467, 110: 464, 142: 462, 130: 459, 143: 459, 136: 458, 191: 456, 75: 455, 

In [ ]:
# What fields do we have?
print(df_aqua.columns.tolist())
print()

# What does the answer field look like?
print(df_aqua["correct"].value_counts())
print()

# How many options per question?
df_aqua["options"].apply(len).value_counts()

['question', 'options', 'rationale', 'correct']

correct
C    122
B    116
A    108
D     97
E     57
Name: count, dtype: int64



,count
options,
5,500


## ReClor
Logical reasoning (substitute for LogiQA)

In [ ]:
df_reclor = explore("metaeval/reclor", name=None, split="train", n=3, sample=500)

  metaeval/reclor  |  config=None


README.md:   0%|          | 0.00/407 [00:00<?, ?B/s]

  Available splits : ['train', 'validation']
  Loading split    : 'train' ...


train.json: 0.00B [00:00, ?B/s]

val.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4638 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]


  ── Full dataset stats ──────────────────────────────────────
  Total rows  : 4638
  Columns     : ['answers', 'context', 'label', 'id_string', 'question']

  [answers]
    dtype      : object
    sample val : array(['Unlike aspirin and other medications that reduce pain and swelling and that are currently available, the new med
    nulls      : 0
    n_unique   : unhashable (list/array column)
    len dist   : {4: 4638}

  [context]
    dtype      : object
    sample val : "In rheumatoid arthritis, the body' s immune system misfunctions by attacking healthy cells in the joints causing the re
    nulls      : 0
    n_unique   : 4628
    len dist   : {361: 26, 445: 25, 317: 23, 391: 23, 358: 21, 430: 21, 341: 20, 398: 20, 466: 20, 441: 20, 334: 19, 421: 19, 350: 19, 351: 19, 382: 18, 458: 18, 429: 18, 316: 18, 370: 18, 432: 18, 483: 18, 438: 18, 359: 18, 383: 18, 425: 18, 392: 18, 328: 18, 427: 17, 418: 17, 414: 17, 374: 17, 461: 17, 385: 17, 456: 17, 356: 17, 371: 17, 368: 17, 387: 1

In [ ]:
print(df_reclor.columns.tolist())
print()
print(df_reclor["label"].value_counts())
print()
df_reclor["answers"].apply(len).value_counts()

['answers', 'context', 'label', 'id_string', 'question']

label
3    138
2    129
1    122
0    111
Name: count, dtype: int64



,count
answers,
4,500


## ARC Easy
Reasoning

In [ ]:
from datasets import get_dataset_config_names
print(get_dataset_config_names("allenai/ai2_arc"))

README.md: 0.00B [00:00, ?B/s]

['ARC-Challenge', 'ARC-Easy']


In [ ]:
df_arc_easy = explore("allenai/ai2_arc", name="ARC-Easy", split="train", n=3, sample=500)

  allenai/ai2_arc  |  config=ARC-Easy
  Available splits : ['train', 'test', 'validation']
  Loading split    : 'train' ...


ARC-Easy/train-00000-of-00001.parquet:   0%|          | 0.00/331k [00:00<?, ?B/s]

ARC-Easy/test-00000-of-00001.parquet:   0%|          | 0.00/346k [00:00<?, ?B/s]

ARC-Easy/validation-00000-of-00001.parqu(…):   0%|          | 0.00/86.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2251 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2376 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/570 [00:00<?, ? examples/s]


  ── Full dataset stats ──────────────────────────────────────
  Total rows  : 2251
  Columns     : ['id', 'question', 'choices', 'answerKey']

  [id]
    dtype      : object
    sample val : 'Mercury_7220990'
    nulls      : 0
    n_unique   : 2251
    len dist   : {15: 911, 17: 459, 14: 351, 22: 147, 16: 125, 13: 104, 21: 63, 19: 44, 12: 17, 8: 16, 18: 14}

  [question]
    dtype      : object
    sample val : 'Which factor will most likely cause a person to develop a fever?'
    nulls      : 0
    n_unique   : 2245
    len dist   : {54: 34, 69: 32, 74: 32, 57: 27, 55: 25, 63: 24, 77: 24, 47: 24, 56: 24, 70: 24, 60: 24, 53: 23, 78: 23, 87: 23, 62: 22, 50: 22, 72: 22, 79: 22, 71: 22, 58: 21, 59: 21, 64: 21, 75: 21, 61: 20, 92: 19, 38: 19, 84: 19, 52: 19, 66: 19, 85: 19, 80: 19, 73: 19, 48: 19, 46: 19, 97: 18, 68: 18, 81: 18, 65: 18, 104: 18, 82: 18, 91: 18, 96: 18, 37: 17, 67: 17, 110: 17, 43: 17, 120: 17, 40: 16, 76: 16, 134: 15, 83: 15, 122: 15, 51: 15, 86: 14, 103: 14, 41: 14, 49

In [ ]:
print("Columns:", df_arc_easy.columns.tolist())
print()
print("Answer key distribution:")
print(df_arc_easy["answerKey"].value_counts())
print()
print("Number of options per question:")
df_arc_easy["choices"].apply(lambda x: len(x["text"])).value_counts()

Columns: ['id', 'question', 'choices', 'answerKey']

Answer key distribution:
answerKey
D    132
B    117
C    116
A    114
4      7
3      7
2      4
1      3
Name: count, dtype: int64

Number of options per question:


,count
choices,
4,498
3,2


## MedQA USMLE


In [ ]:
df_medqa = explore("GBaker/MedQA-USMLE-4-options", name=None, split="train", n=3, sample=500)

  GBaker/MedQA-USMLE-4-options  |  config=None


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

  Available splits : ['train', 'test']
  Loading split    : 'train' ...


phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]


  ── Full dataset stats ──────────────────────────────────────
  Total rows  : 10178
  Columns     : ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases']

  [question]
    dtype      : object
    sample val : 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ag
    nulls      : 0
    n_unique   : 10176
    len dist   : {633: 26, 532: 24, 576: 24, 678: 24, 624: 23, 755: 23, 555: 23, 601: 23, 535: 22, 700: 22, 638: 22, 652: 22, 778: 21, 545: 21, 626: 21, 740: 21, 542: 21, 717: 21, 666: 21, 692: 21, 695: 21, 579: 21, 699: 21, 475: 21, 512: 20, 546: 20, 756: 20, 674: 20, 655: 20, 524: 20, 726: 20, 707: 20, 605: 20, 693: 20, 432: 20, 784: 19, 617: 19, 809: 19, 683: 19, 609: 19, 690: 19, 815: 19, 704: 19, 688: 19, 639: 19, 565: 19, 574: 19, 722: 19, 648: 19, 541: 19, 623: 19, 493: 19, 548: 19, 822: 19, 727: 19, 882: 19, 568: 19, 711: 18, 572: 18, 575: 18, 879: 18, 714: 18, 553: 18, 600: 18, 491: 1

In [ ]:
print("Columns:", df_medqa.columns.tolist())
print()
print("Answer distribution:")
print(df_medqa["answer_idx"].value_counts())
print()
print("Number of options per question:")
df_medqa["options"].apply(lambda x: len(x)).value_counts()
print()
print("Meta info distribution:")
print(df_medqa["meta_info"].value_counts())

Columns: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases']

Answer distribution:
answer_idx
B    131
D    127
A    124
C    118
Name: count, dtype: int64

Number of options per question:

Meta info distribution:
meta_info
step1      273
step2&3    227
Name: count, dtype: int64


## SciQA

In [ ]:
df_sciq = explore("allenai/sciq", name=None, split="train", n=3, sample=500)

  allenai/sciq  |  config=None


README.md: 0.00B [00:00, ?B/s]

  Available splits : ['train', 'validation', 'test']
  Loading split    : 'train' ...


data/train-00000-of-00001.parquet:   0%|          | 0.00/3.99M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/339k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/343k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11679 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]


  ── Full dataset stats ──────────────────────────────────────
  Total rows  : 11679
  Columns     : ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support']

  [question]
    dtype      : object
    sample val : 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?'
    nulls      : 0
    n_unique   : 11609
    len dist   : {61: 213, 64: 199, 51: 192, 69: 189, 59: 185, 58: 182, 57: 182, 52: 182, 56: 181, 55: 180, 50: 179, 54: 178, 66: 177, 49: 175, 62: 175, 67: 172, 71: 172, 60: 170, 68: 170, 53: 170, 65: 167, 73: 165, 72: 162, 63: 162, 48: 162, 77: 158, 70: 156, 76: 154, 44: 151, 74: 148, 83: 148, 46: 147, 82: 147, 45: 147, 81: 144, 47: 144, 79: 143, 75: 142, 42: 138, 43: 137, 85: 135, 80: 134, 78: 133, 87: 130, 90: 129, 84: 128, 86: 124, 89: 119, 91: 118, 88: 111, 92: 109, 95: 107, 41: 106, 39: 102, 38: 98, 40: 96, 98: 95, 93: 92, 99: 91, 102: 89, 97: 89, 94: 88, 104: 87, 36: 79, 96: 76, 100: 75, 106: 73, 101: 73, 3

In [ ]:
print("Columns:", df_sciq.columns.tolist())
print()

Columns: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support']



# Dataset Gathering

## Analyzing the columns and datatypes

In [ ]:
import pandas as pd
from pprint import pformat

# Make pandas less aggressive about truncation
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

datasets = {
    "MMLU": df_mmlu,
    "AQUA-RAT": df_aqua,
    "ReClor": df_reclor,
    "MedQA": df_medqa,
    "ARC-Easy": df_arc_easy,
    "SciQ": df_sciq,
}

def print_full_sample(df, name):
    print("=" * 120)
    print(f"{name} Columns: {df.columns.tolist()}")
    print()

    print(f"{name} dtypes:")
    print(df.dtypes.to_string())
    print()

    if len(df) == 0:
        print(f"{name} sample: <empty dataframe>")
        return

    sample = df.iloc[0]
    print(f"{name} first sample, full values:")
    for col in df.columns:
        value = sample[col]
        print(f"\n[{col}]")
        print(f"  pandas dtype: {df[col].dtype}")
        print(f"  python type : {type(value)}")
        print(f"  value       : {pformat(value, width=200, compact=False, sort_dicts=False)}")

for name, df in datasets.items():
    print_full_sample(df, name)
    print()

MMLU Columns: ['question', 'subject', 'choices', 'answer']

MMLU dtypes:
question    object
subject     object
choices     object
answer       int64

MMLU first sample, full values:

[question]
  pandas dtype: object
  python type : <class 'str'>
  value       : ("This is Bruce's Noodle House. We have different kinds of noodles. A large bowl of noodles with mutton  is only 4 dollars, and 3 dollars for a medium  bowl. Each bowl of beef noodles is just 3.5 "
 'dollars. And a large bowl of chicken noodles is 2 dollars. Each bowl of pork noodles is just 3.5 dollars. One dollar is for a large bowl of vegetable noodles. Come and enjoy the delicious noodles '
 'here! If you order any meat noodles, fruit is free . If you are too busy to come. Please call us at 2888998, and you will get the food in half an hour. Our Noodle House is open for 24 hours a day, '
 "from Monday afternoon to Sunday. Tom's family would like two large bowls of chicken noodles and a medium bowl of mutton noodles. They wi

## Normalize and Combine them

In [ ]:
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

datasets = {
    "MMLU": df_mmlu,
    "AQUA-RAT": df_aqua,
    "ReClor": df_reclor,
    "MedQA": df_medqa,
    "ARC-Easy": df_arc_easy,
    "SciQ": df_sciq,
}

def to_list(x):
    if x is None:
        return None
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]

def idx_to_label(i):
    return chr(ord("A") + int(i))

def clean_text(x):
    if x is None:
        return None
    return str(x).strip()

def parse_labeled_option(option):
    """
    Parses strings like:
      A)120960
      B. 42
      C: text
    Returns (label, text). If no label prefix is found, returns (None, original_text).
    """
    s = clean_text(option)
    m = re.match(r"^\s*([A-Z])[\)\.\:\-]?\s*(.*)\s*$", s)
    if m and m.group(1) in {"A", "B", "C", "D", "E", "F", "G", "H"}:
        return m.group(1), m.group(2)
    return None, s

def make_choice_payload(labels, texts):
    labels = [clean_text(x) for x in labels]
    texts = [clean_text(x) for x in texts]
    return labels, texts

def normalize_mmlu(df):
    rows = []
    for _, r in df.iterrows():
        texts = to_list(r["choices"])
        labels = [idx_to_label(i) for i in range(len(texts))]
        gold_index = int(r["answer"])
        gold_label = labels[gold_index] if gold_index < len(labels) else None
        gold_text = texts[gold_index] if gold_index < len(texts) else None

        rows.append({
            "source": "MMLU",
            "task_type": "mcq",
            "source_id": None,
            "question": clean_text(r["question"]),
            "context": None,
            "choice_labels": labels,
            "choice_texts": texts,
            "gold_label": gold_label,
            "gold_index": gold_index,
            "gold_text": gold_text,
            "explanation": None,
            "subject": clean_text(r["subject"]),
            "meta": {}
        })
    return pd.DataFrame(rows)

def normalize_aqua(df):
    rows = []
    for _, r in df.iterrows():
        raw_options = to_list(r["options"])
        parsed = [parse_labeled_option(opt) for opt in raw_options]
        labels = [p[0] if p[0] is not None else idx_to_label(i) for i, p in enumerate(parsed)]
        texts = [p[1] for p in parsed]

        gold_label = clean_text(r["correct"])
        gold_index = labels.index(gold_label) if gold_label in labels else None
        gold_text = texts[gold_index] if gold_index is not None and gold_index < len(texts) else None

        rows.append({
            "source": "AQUA-RAT",
            "task_type": "mcq_rationale",
            "source_id": None,
            "question": clean_text(r["question"]),
            "context": None,
            "choice_labels": labels,
            "choice_texts": texts,
            "gold_label": gold_label,
            "gold_index": gold_index,
            "gold_text": gold_text,
            "explanation": clean_text(r["rationale"]),
            "subject": None,
            "meta": {}
        })
    return pd.DataFrame(rows)

def normalize_reclor(df):
    rows = []
    for _, r in df.iterrows():
        texts = to_list(r["answers"])
        labels = [idx_to_label(i) for i in range(len(texts))]
        gold_index = int(r["label"])
        gold_label = labels[gold_index] if gold_index < len(labels) else None
        gold_text = texts[gold_index] if gold_index < len(texts) else None

        rows.append({
            "source": "ReClor",
            "task_type": "reading_comp_mcq",
            "source_id": clean_text(r["id_string"]),
            "question": clean_text(r["question"]),
            "context": clean_text(r["context"]),
            "choice_labels": labels,
            "choice_texts": texts,
            "gold_label": gold_label,
            "gold_index": gold_index,
            "gold_text": gold_text,
            "explanation": None,
            "subject": None,
            "meta": {}
        })
    return pd.DataFrame(rows)

def normalize_medqa(df):
    rows = []
    for _, r in df.iterrows():
        opt = r["options"]
        if isinstance(opt, dict):
            labels = [clean_text(k) for k in opt.keys()]
            texts = [clean_text(v) for v in opt.values()]
        else:
            labels, texts = make_choice_payload([], to_list(opt))

        gold_label = clean_text(r["answer_idx"])
        gold_index = labels.index(gold_label) if gold_label in labels else None
        gold_text = clean_text(r["answer"])

        rows.append({
            "source": "MedQA",
            "task_type": "mcq_medical",
            "source_id": None,
            "question": clean_text(r["question"]),
            "context": clean_text(r["meta_info"]),
            "choice_labels": labels,
            "choice_texts": texts,
            "gold_label": gold_label,
            "gold_index": gold_index,
            "gold_text": gold_text,
            "explanation": None,
            "subject": None,
            "meta": {
                "metamap_phrases": to_list(r["metamap_phrases"])
            }
        })
    return pd.DataFrame(rows)

def normalize_arc_easy(df):
    rows = []
    for _, r in df.iterrows():
        choices = r["choices"]
        if isinstance(choices, dict):
            labels = to_list(choices.get("label"))
            texts = to_list(choices.get("text"))
        else:
            texts = to_list(choices)
            labels = [idx_to_label(i) for i in range(len(texts))]

        gold_label = clean_text(r["answerKey"])
        gold_index = labels.index(gold_label) if gold_label in labels else None
        gold_text = texts[gold_index] if gold_index is not None and gold_index < len(texts) else None

        rows.append({
            "source": "ARC-Easy",
            "task_type": "mcq_science",
            "source_id": clean_text(r["id"]),
            "question": clean_text(r["question"]),
            "context": None,
            "choice_labels": labels,
            "choice_texts": texts,
            "gold_label": gold_label,
            "gold_index": gold_index,
            "gold_text": gold_text,
            "explanation": None,
            "subject": None,
            "meta": {}
        })
    return pd.DataFrame(rows)

def normalize_sciq(df):
    rows = []
    for _, r in df.iterrows():
        texts = [
            clean_text(r["correct_answer"]),
            clean_text(r["distractor1"]),
            clean_text(r["distractor2"]),
            clean_text(r["distractor3"]),
        ]
        labels = [idx_to_label(i) for i in range(len(texts))]
        gold_index = 0
        gold_label = labels[gold_index]
        gold_text = texts[gold_index]

        rows.append({
            "source": "SciQ",
            "task_type": "mcq_science",
            "source_id": None,
            "question": clean_text(r["question"]),
            "context": clean_text(r["support"]),
            "choice_labels": labels,
            "choice_texts": texts,
            "gold_label": gold_label,
            "gold_index": gold_index,
            "gold_text": gold_text,
            "explanation": None,
            "subject": None,
            "meta": {"choice_order": "constructed_from_correct_plus_distractors"}
        })
    return pd.DataFrame(rows)

normalized_frames = [
    normalize_mmlu(df_mmlu),
    normalize_aqua(df_aqua),
    normalize_reclor(df_reclor),
    normalize_medqa(df_medqa),
    normalize_arc_easy(df_arc_easy),
    normalize_sciq(df_sciq),
]

combined = pd.concat(normalized_frames, ignore_index=True)

# Optional sanity checks
print("Combined shape:", combined.shape)
print(combined["source"].value_counts(dropna=False).to_string())
print()

for src in combined["source"].unique():
    row = combined[combined["source"] == src].iloc[0]
    print("=" * 120)
    print(src)
    for col in ["question", "context", "choice_labels", "choice_texts", "gold_label", "gold_index", "gold_text", "explanation", "subject", "meta"]:
        print(f"\n[{col}]")
        print(row[col])

Combined shape: (3000, 13)
source
MMLU        500
AQUA-RAT    500
ReClor      500
MedQA       500
ARC-Easy    500
SciQ        500

MMLU

[question]
This is Bruce's Noodle House. We have different kinds of noodles. A large bowl of noodles with mutton  is only 4 dollars, and 3 dollars for a medium  bowl. Each bowl of beef noodles is just 3.5 dollars. And a large bowl of chicken noodles is 2 dollars. Each bowl of pork noodles is just 3.5 dollars. One dollar is for a large bowl of vegetable noodles. Come and enjoy the delicious noodles here! If you order any meat noodles, fruit is free . If you are too busy to come. Please call us at 2888998, and you will get the food in half an hour. Our Noodle House is open for 24 hours a day, from Monday afternoon to Sunday. Tom's family would like two large bowls of chicken noodles and a medium bowl of mutton noodles. They will pay    _   for the food.

[context]
None

[choice_labels]
['A', 'B', 'C', 'D']

[choice_texts]
['$6', '$7', '$5', '$8']

[gold

## Verification

In [ ]:
import pandas as pd

print("=" * 80)
print("BASIC INFO")
print("=" * 80)
print("Shape:", combined.shape)
print()
print("Source distribution:")
print(combined["source"].value_counts().to_string())
print()

# -----------------------------
# Missing values
# -----------------------------
print("=" * 80)
print("MISSING VALUES")
print("=" * 80)
print(combined.isna().sum().to_string())
print()

# -----------------------------
# Choice consistency
# -----------------------------
def check_choices(row):
    labels = row["choice_labels"]
    texts = row["choice_texts"]
    if not isinstance(labels, list) or not isinstance(texts, list):
        return False
    if len(labels) != len(texts):
        return False
    if len(labels) < 2:
        return False
    return True

combined["valid_choices"] = combined.apply(check_choices, axis=1)

print("=" * 80)
print("CHOICE STRUCTURE")
print("=" * 80)
print("Invalid choice rows:", (~combined["valid_choices"]).sum())
print()

# -----------------------------
# Gold alignment checks
# -----------------------------
def check_gold(row):
    labels = row["choice_labels"]
    texts = row["choice_texts"]
    g_label = row["gold_label"]
    g_index = row["gold_index"]
    g_text = row["gold_text"]

    try:
        if g_index is not None:
            if g_index >= len(texts):
                return False
            if texts[g_index] != g_text:
                return False

        if g_label is not None and g_label in labels:
            idx = labels.index(g_label)
            if texts[idx] != g_text:
                return False

        return True
    except Exception:
        return False

combined["valid_gold"] = combined.apply(check_gold, axis=1)

print("=" * 80)
print("GOLD ALIGNMENT")
print("=" * 80)
print("Invalid gold rows:", (~combined["valid_gold"]).sum())
print()

# -----------------------------
# Empty / bad questions
# -----------------------------
combined["bad_question"] = combined["question"].isna() | (combined["question"].str.strip() == "")

print("=" * 80)
print("QUESTION QUALITY")
print("=" * 80)
print("Empty questions:", combined["bad_question"].sum())
print()

# -----------------------------
# Summary of all failures
# -----------------------------
combined["any_issue"] = ~(
    combined["valid_choices"] &
    combined["valid_gold"] &
    ~combined["bad_question"]
)

print("=" * 80)
print("OVERALL DATA HEALTH")
print("=" * 80)
print("Rows with any issue:", combined["any_issue"].sum())
print()

# Optional: inspect bad rows
bad_rows = combined[combined["any_issue"]]
if len(bad_rows) > 0:
    print("Sample problematic row:")
    print(bad_rows.iloc[0].to_dict())

BASIC INFO
Shape: (3000, 13)

Source distribution:
source
MMLU        500
AQUA-RAT    500
ReClor      500
MedQA       500
ARC-Easy    500
SciQ        500

MISSING VALUES
source              0
task_type           0
source_id        2000
question            0
context          1500
choice_labels       0
choice_texts        0
gold_label          0
gold_index          0
gold_text           0
explanation      2500
subject          2500
meta                0

CHOICE STRUCTURE
Invalid choice rows: 0

GOLD ALIGNMENT
Invalid gold rows: 0

QUESTION QUALITY
Empty questions: 0

OVERALL DATA HEALTH
Rows with any issue: 0



In [ ]:
# Drop verification columns if present
cols_to_drop = [
    "valid_choices",
    "valid_gold",
    "bad_question",
    "any_issue"
]

clean_combined = combined.drop(columns=[c for c in cols_to_drop if c in combined.columns])

# Save as pickle (preserves Python objects like lists/dicts cleanly)
clean_combined.to_pickle("combined_dataset.pkl")

# Save as JSON (more portable, but larger)
clean_combined.to_json("combined_dataset.json", orient="records", indent=2)

# Optional: CSV (not ideal for nested fields, but usable)
clean_combined.to_csv("combined_dataset.csv", index=False)

print("Saved:")
print(" - combined_dataset.pkl")
print(" - combined_dataset.json")
print(" - combined_dataset.csv")

Saved:
 - combined_dataset.pkl
 - combined_dataset.json
 - combined_dataset.csv


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(clean_combined, test_size=600, random_state=42, stratify=clean_combined["source"])
val_df, test_df = train_test_split(temp_df, test_size=300, random_state=42, stratify=temp_df["source"])

print(f"Train: {len(train_df)}")
print(train_df["source"].value_counts().to_string())
print(f"\nVal: {len(val_df)}")
print(val_df["source"].value_counts().to_string())
print(f"\nTest: {len(test_df)}")
print(test_df["source"].value_counts().to_string())

train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

from google.colab import files
files.download("train.csv")
files.download("validation.csv")
files.download("test.csv")

Train: 2400
source
MedQA       400
AQUA-RAT    400
ReClor      400
MMLU        400
ARC-Easy    400
SciQ        400

Val: 300
source
ReClor      50
MedQA       50
SciQ        50
ARC-Easy    50
MMLU        50
AQUA-RAT    50

Test: 300
source
SciQ        50
MedQA       50
MMLU        50
ARC-Easy    50
ReClor      50
AQUA-RAT    50


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>